# Hosting an Existing Docker Image in Amazon Bedrock AgentCore Runtime

## Overview

In this tutorial we will learn how to deploy an existing Docker image to Amazon Bedrock AgentCore Runtime, using the boto3 SDK directly.

We will focus on deploying a pre-built container image from Amazon ECR. For deploying using the starter toolkit check [here](../01-strands-with-bedrock-model) and for LangGraph with Amazon Bedrock model check [here](../02-langgraph-with-bedrock-model).


### Tutorial Details


| Information         | Details                                                                                      |
|:--------------------|:---------------------------------------------------------------------------------------------|
| Tutorial type       | Conversational                                                                               |
| Agent type          | Single                                                                                       |
| Agentic Framework   | Strands Agents (pre-built in Docker image)                                                   |
| LLM model           | Anthropic Claude Haiku 4.5                                                                    |
| Tutorial components | Building Docker image, pushing to ECR, deploying to AgentCore Runtime using boto3            |
| Tutorial vertical   | Cross-vertical                                                                               |
| Example complexity  | Easy                                                                                         |
| SDK used            | boto3                                                                                        |

### Tutorial Architecture

In this tutorial we will describe how to build a Docker image containing a Strands Agent, push it to Amazon ECR, and deploy it to AgentCore Runtime using the `create_agent_runtime` API.

For demonstration purposes, we will  use the same Strands Agent from [01-strands-with-bedrock-model](../01-strands-with-bedrock-model) with two tools: `calculator` and `weather`.

### Tutorial Key Features

* Building and pushing Docker images for AgentCore Runtime
* Creating an IAM execution role for AgentCore Runtime
* Deploying to AgentCore Runtime using boto3
* Invoking and managing the deployed agent


## Prerequisites

To execute this tutorial you will need:
* Python 3.10+
* AWS credentials
* Docker installed and running
* AWS CLI installed

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
import boto3
import json
import time

REGION = "us-west-2"  # Change to your preferred region

sts_client = boto3.client('sts', region_name=REGION)
ACCOUNT_ID = sts_client.get_caller_identity()['Account']
print(f"ACCOUNT_ID: {ACCOUNT_ID}")
print(f"REGION: {REGION}")

## Creating the agent code

Before we build our Docker image, let's create the agent code that will be packaged into it. This is the same Strands Agent from [01-strands-with-bedrock-model](../01-strands-with-bedrock-model), already prepared for deployment with `BedrockAgentCoreApp`.

For production agentic applications we will need to decouple the agent creation process from the agent invocation one. With AgentCore Runtime, we decorate the invocation part of our agent with the `@app.entrypoint` decorator and have it as the entry point for our runtime.

If you already have an agent image in ECR, you can skip ahead to [Deploying to AgentCore Runtime](#deploying-to-agentcore-runtime).

In [ ]:
%%writefile strands_claude.py
from strands import Agent, tool
from strands_tools import calculator # Import the calculator tool
import argparse
import json
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands.models import BedrockModel

app = BedrockAgentCoreApp()

# Create a custom tool 
@tool
def weather():
    """ Get weather """ # Dummy implementation
    return "sunny"


model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
model = BedrockModel(
    model_id=model_id,
)
agent = Agent(
    model=model,
    tools=[calculator, weather],
    system_prompt="You're a helpful assistant. You can do simple math calculation, and tell the weather."
)

@app.entrypoint
def strands_agent_bedrock(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    print("User input:", user_input)
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

## What happens behind the scenes?

When you use `BedrockAgentCoreApp`, it automatically:

* Creates an HTTP server that listens on the port 8080
* Implements the required `/invocations` endpoint for processing the agent's requirements
* Implements the `/ping` endpoint for health checks (very important for asynchronous agents)
* Handles proper content types and response formats
* Manages error handling according to the AWS standards

## Creating the Dockerfile

AgentCore Runtime expects your container to listen on port 8080, run as a non-root user (UID 1000), and expose the `/invocations` and `/ping` endpoints. The `BedrockAgentCoreApp` handles the endpoints automatically, so we just need to set up the container correctly.

In [ ]:
%%writefile Dockerfile
FROM public.ecr.aws/docker/library/python:3.11-slim
WORKDIR /app

COPY requirements.txt requirements.txt
RUN pip install -r requirements.txt

# Create non-root user (required by AgentCore Runtime)
RUN useradd -m -u 1000 bedrock_agentcore
USER bedrock_agentcore

EXPOSE 8080

COPY . .

CMD ["python", "strands_claude.py"]

## Building and pushing the Docker image to ECR

Now let's create an ECR repository, build the Docker image, and push it. This is what your CI/CD pipeline would typically handle for you.

On x86 instances (e.g., SageMaker), the arm64 cross-compilation via QEMU can take over an hour. On a native arm64 (Graviton) instance it takes a couple minutes.



In [ ]:
# Authenticate Docker with ECR
!aws ecr get-login-password --region {REGION}| docker login --username AWS --password-stdin {ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com

In [ ]:
# Enable arm64 emulation **(only needed on x86 instances)**
!docker run --rm --privileged multiarch/qemu-user-static --reset -p yes

In [ ]:
# Create ECR repository
ECR_REPO_NAME = "strands-agentcore-existing-image"

ecr_client = boto3.client('ecr', region_name=REGION)

# Create ECR repository (skip if it already exists)
try:
    ecr_client.create_repository(repositoryName=ECR_REPO_NAME)
    print(f"ECR repository '{ECR_REPO_NAME}' created")
except ecr_client.exceptions.RepositoryAlreadyExistsException:
    print(f"ECR repository '{ECR_REPO_NAME}' already exists")

IMAGE_URI = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{ECR_REPO_NAME}:latest"
print(f"Image URI: {IMAGE_URI}")

In [ ]:
# Build the Docker image (AgentCore Runtime requires arm64). The arm64 cross-compilation via QEMU can take over an hour
!docker buildx build --platform linux/arm64 -t strands-agentcore-existing-image:latest . 

In [ ]:
# Tag ECR
!docker tag strands-agentcore-existing-image:latest {ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/strands-agentcore-existing-image:latest


In [ ]:
# Push to ECR
!docker push {ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/strands-agentcore-existing-image:latest

In [ ]:
# After running the terminal commands above, verify the image was pushed
resp = ecr_client.describe_images(repositoryName=ECR_REPO_NAME, imageIds=[{'imageTag': 'latest'}])
print(f"Image found in ECR: {resp['imageDetails'][0]['imageTags']}")
print(f"   Pushed at: {resp['imageDetails'][0]['imagePushedAt']}")

## Creating the IAM execution role

AgentCore Runtime needs an IAM execution role to pull the container image from ECR and invoke Amazon Bedrock models. If you already have an execution role (e.g., from a previous tutorial), you can set `ROLE_ARN` directly and skip the role creation.

```python
# If you already have a role, set it directly:
# ROLE_ARN = "arn:aws:iam::<YOUR_ACCOUNT_ID>:role/AgentCoreExecutionRole"
```

In [ ]:
iam_client = boto3.client('iam')

ROLE_NAME = "AgentCoreExistingImageExecutionRole"

# Trust policy: Allow bedrock-agentcore service to assume role
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {"aws:SourceAccount": ACCOUNT_ID},
                "ArnLike": {
                    "aws:SourceArn": f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:runtime/*"
                }
            }
        }
    ]
}

# Permissions policy for Runtime
permissions_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "CloudWatchLogs",
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:PutLogEvents"
            ],
            "Resource": f"arn:aws:logs:{REGION}:{ACCOUNT_ID}:log-group:/aws/bedrock-agentcore/runtime/*"
        },
        {
            "Sid": "ECRAccess",
            "Effect": "Allow",
            "Action": [
                "ecr:GetAuthorizationToken",
                "ecr:BatchGetImage",
                "ecr:GetDownloadUrlForLayer"
            ],
            "Resource": "*"
        },
        {
            "Sid": "BedrockModels",
            "Effect": "Allow",
            "Action": [
                "bedrock:InvokeModel",
                "bedrock:InvokeModelWithResponseStream"
            ],
            "Resource": [
                f"arn:aws:bedrock:{REGION}::foundation-model/*",
                f"arn:aws:bedrock:*:{ACCOUNT_ID}:inference-profile/*",
                 "arn:aws:bedrock:*::foundation-model/*"
            ]
        }
    ]
}

# Create the role
try:
    role_response = iam_client.create_role(
        RoleName=ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description="Execution role for AgentCore Runtime with existing Docker image"
    )
    ROLE_ARN = role_response['Role']['Arn']
    print(f"Role created: {ROLE_ARN}")
except iam_client.exceptions.EntityAlreadyExistsException:
    ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{ROLE_NAME}"
    print(f"Role already exists: {ROLE_ARN}")

# Attach the permissions policy
iam_client.put_role_policy(
    RoleName=ROLE_NAME,
    PolicyName="AgentCoreRuntimePermissions",
    PolicyDocument=json.dumps(permissions_policy)
)
print("Permissions policy attached")

# Wait for IAM propagation
print("Waiting for IAM role propagation...")
time.sleep(10)

<a id="deploying-to-agentcore-runtime"></a>
## Deploying to AgentCore Runtime

The `CreateAgentRuntime` operation supports comprehensive configuration options, letting you specify container images, environment variables and encryption settings. You can also configure protocol settings (HTTP, MCP) and authorization mechanisms to control how your clients communicate with the agent. 

**Note:** Operations best practice is to package code as container and push to ECR using CI/CD pipelines and IaC

In this tutorial we will use the boto3 `bedrock-agentcore-control` client to create the runtime directly, pointing at our existing ECR image.

### Creating the AgentCore Runtime

Let's create the AgentCore Runtime with our existing Docker image. We pass the ECR image URI directly to the `containerConfiguration`.

In [ ]:
AGENT_RUNTIME_NAME = "strands_existing_image_getting_started"

control_client = boto3.client('bedrock-agentcore-control', region_name=REGION)

response = control_client.create_agent_runtime(
    agentRuntimeName=AGENT_RUNTIME_NAME,
    agentRuntimeArtifact={
        'containerConfiguration': {
            'containerUri': IMAGE_URI
        }
    },
    networkConfiguration={"networkMode": "PUBLIC"},
    roleArn=ROLE_ARN
)

runtime_arn = response['agentRuntimeArn']
runtime_id = runtime_arn.split('/')[-1]

print(f"AgentCore Runtime created!")
print(f"ARN: {runtime_arn}")
print(f"Runtime ID: {runtime_id}")

### Checking for the AgentCore Runtime Status
Now that we've deployed the AgentCore Runtime, let's check for it's deployment status

In [ ]:
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']

status = None
while status not in end_status:
    time.sleep(10)
    status_response = control_client.get_agent_runtime(agentRuntimeId=runtime_id)
    status = status_response.get('status')
    print(status)
status

### Creating the DEFAULT endpoint

Once the runtime is ready, we need to create a DEFAULT endpoint. This is the endpoint that clients will use to invoke the agent.

In [ ]:
try:
    endpoint_response = control_client.create_agent_runtime_endpoint(
        agentRuntimeId=runtime_id,
        name="DEFAULT"
    )
    print(f"DEFAULT endpoint created!")
    print(f"Endpoint ARN: {endpoint_response['agentRuntimeEndpointArn']}")
except Exception as e:
    if "already exists" in str(e):
        print(f"DEFAULT endpoint already exists")
    else:
        raise e

### Invoking AgentCore Runtime

Now that your AgentCore Runtime was created you can invoke it with any AWS SDK. For instance, you can use the boto3 `invoke_agent_runtime` method for it.

In [ ]:
from IPython.display import Markdown, display

agentcore_client = boto3.client(
    'bedrock-agentcore',
    region_name=REGION
)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=runtime_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "How is the weather now?"})
)

# Capture the runtime session ID for lifecycle management
runtime_session_id = boto3_response.get('runtimeSessionId')
print(f"Runtime Session ID: {runtime_session_id}")

if "text/event-stream" in boto3_response.get("contentType", ""):
    content = []
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                print(line)
                content.append(line)
    display(Markdown("\n".join(content)))
else:
    try:
        events = []
        for event in boto3_response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]
    display(Markdown(json.loads(events[0].decode("utf-8"))))

## Cleanup

Let's now clean up the AgentCore Runtime and associated resources. We delete the runtime first to avoid undesired costs, then clean up supporting resources like ECR repositories.

In [ ]:
runtime_arn, runtime_id, ECR_REPO_NAME

In [ ]:
# --- Stop active sessions to release microVM resources ---
import boto3

agentcore_client = boto3.client('bedrock-agentcore', region_name=REGION)
control_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
ecr_client = boto3.client('ecr', region_name=REGION)

# Stop the active session to release its microVM resources
# In production, this is how you end individual user sessions while keeping the runtime alive
# AgentCore Runtime costs are based on vCPU and Memory — stopping sessions avoids undesired costs
if 'runtime_session_id' in locals() and runtime_session_id:
    try:
        agentcore_client.stop_runtime_session(
            agentRuntimeArn=runtime_arn,
            runtimeSessionId=runtime_session_id,
            qualifier='DEFAULT'
        )
        print(f"Session '{runtime_session_id}' stopped")
    except Exception as e:
        print(f"Failed to stop session '{runtime_session_id}': {e}")


try:
    endpoints_response = control_client.list_agent_runtime_endpoints(agentRuntimeId=runtime_id)
    for ep in endpoints_response.get('agentRuntimeEndpoints', []):
        ep_id = ep['agentRuntimeEndpointId']
        control_client.delete_agent_runtime_endpoint(
            agentRuntimeEndpointId=ep_id,
            agentRuntimeId=runtime_id
        )
        print(f"Endpoint '{ep_id}' deleted")
    time.sleep(30)
except Exception as e:
    print(f"Failed to delete endpoint: {e}")

# --- Delete the runtime ---
try:
    control_client.delete_agent_runtime(
        agentRuntimeId=runtime_id,
    )
    print(f"Runtime '{runtime_id}' deleted")
except Exception as e:
    print(f"Failed to delete runtime: {e}")

# --- Delete ECR repository ---
try:
    ecr_client.delete_repository(
        repositoryName=ECR_REPO_NAME,
        force=True
    )
    print(f"ECR repository '{ECR_REPO_NAME}' deleted")
except Exception as e:
    print(f"Failed to delete ECR repository: {e}")

# --- Delete IAM role ---
try:
    iam_client = boto3.client('iam')
    iam_client.delete_role_policy(
        RoleName=ROLE_NAME,
        PolicyName="AgentCoreRuntimePermissions"
    )
    iam_client.delete_role(RoleName=ROLE_NAME)
    print(f"IAM role '{ROLE_NAME}' deleted")
except Exception as e:
    print(f"Failed to delete IAM role: {e}")

# Congratulations!